# overnight — 전체 실행 한 방 (diag3 → seed 학습 → eval → 집계)

**Run All 걸어 놓고 자면 되는 노트북이다.** 위에서 아래로 한 번에 돈다.

| 단계 | 무엇 | GPU | 시간 |
|---|---|---|---|
| **A** | diag3 — `b_k` 의 어떤 성질이 일하는가 (학습 없음) | 0 하나 | 수 분 |
| **B** | seed 1·2 학습 — `bimamba_pure`, `acm2` × K=100 = **4잡** | 0~3 | **~8h** |
| **C** | eval — seed 별 2셀 (K=100, s=10, n=500) | 0~1 | ~2h × 2 seed |
| **D** | 집계 — gap 평균 ± 표준편차, 요약 파일 저장 | — | 즉시 |

총 **~12h 안팎**. B 가 대부분이다.

## 왜 seed 재현인가

논문 헤드라인 `BiMamba 65.0 vs ACM2 56.8 = +8.2` 가 **seed 0 하나**다. 리뷰어가 제일
먼저 묻는 곳이고, diag 로 메커니즘을 규명해도 이게 안 받쳐 주면 소용없다.
**코드 작업이 0 이고 잃을 게 없는 유일한 실험**이라 제일 먼저 건다.

## 무인 실행 안전장치

- **사전 점검(셀 4)이 틀리면 즉시 중단한다.** 순수 BiMamba 가 `--use_chunk_pairs`
  를 달고 있거나 `--policy.sscp_enabled=false` 가 없으면 8시간을 날리므로 **여기서만 멈춘다.**
- **그 뒤 단계는 서로 격리돼 있다.** A 가 죽어도 B 는 돈다. B 가 중간에 죽어도 C·D 는
  가진 것만으로 진행한다.
- **각 단계가 끝날 때마다 상태를 파일로 쓴다** (`outputs/final/share/overnight/`).
  브라우저를 닫으면 셀 출력이 사라질 수 있으니 **아침에는 이 파일부터 본다.**
- **다시 실행해도 안전하다.** 150k 인 학습은 skip, 중단된 건 resume, 끝난 eval 은 skip.

> ⚠️ 이 노트북은 `exp5_tonight.py` 와 `diag3_bk_variants.py` 를 부르기만 한다.
> 잡 정의·스킵·집계는 전부 거기 있다.

## 0) 부팅

In [ ]:
import json, os, subprocess, sys, time, traceback
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
REPO = _r

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

STAMP = time.strftime('%Y%m%d_%H%M')
SHARE = Path(os.environ.get('LEROBOT_OUTPUT',
                            Path.home() / 'lerobot_project' / 'outputs')) / 'final' / 'share' / 'overnight'
SHARE.mkdir(parents=True, exist_ok=True)
STATUS_PATH = SHARE / f'status_{STAMP}.json'
STATUS = {'stamp': STAMP, 'stages': {}}

def save_status():
    STATUS_PATH.write_text(json.dumps(STATUS, ensure_ascii=False, indent=1, default=str),
                           encoding='utf-8')

def stage(name, fn):
    """단계를 격리해서 돌린다. 죽어도 다음 단계는 계속 간다."""
    t0 = time.time()
    print('\n' + '=' * 70 + f'\n[{name}] 시작 {time.strftime("%H:%M:%S")}\n' + '=' * 70)
    try:
        out = fn()
        STATUS['stages'][name] = {'ok': True, 'sec': round(time.time() - t0)}
        return out
    except Exception as e:
        traceback.print_exc()
        print(f'\n!! [{name}] 실패 — {type(e).__name__}: {e}')
        print('   이 단계만 건너뛰고 다음으로 간다.')
        STATUS['stages'][name] = {'ok': False, 'sec': round(time.time() - t0),
                                  'error': f'{type(e).__name__}: {e}'}
        return None
    finally:
        save_status()
        print(f'[{name}] 끝 — {STATUS["stages"][name]}')

print('repo      :', REPO)
print('TASK      :', X.TASK, '  기본 SEED:', X.SEED)
print('MAIN_STRIDE:', X.MAIN_STRIDE, ' N_EP:', X.N_EP, '(task 당 -> overall 500)')
print('GPU       :', v23.available_gpus())
print('상태 파일 :', STATUS_PATH)

## 1) 설정

In [ ]:
GPUS     = [0, 1, 2, 3]            # GPU 4개
K        = 100                     # 헤드라인 셀
SEEDS    = [1, 2]                  # 새로 돌릴 seed (0 은 이미 있음)
VARIANTS = ['bimamba', 'acm2']     # exp5 변형 이름. bimamba -> tag 'bimamba_pure'

DIAG_GPU   = '0'                   # diag3 은 학습 시작 전에 끝나므로 아무 GPU 나 된다
DIAG_SEED  = 0                     # diag3 이 여는 체크포인트 seed
DIAG_STEP  = 150_000

TAGS = [X.tag_of(v, K) for v in VARIANTS]
JOBS = [(t, s, X.TASK) for s in SEEDS for t in TAGS]

print('태그 :', TAGS)
print('잡   :', len(JOBS), '개')
for j in JOBS:
    print('   ', j)
STATUS['config'] = {'K': K, 'seeds': SEEDS, 'variants': VARIANTS, 'jobs': [list(j) for j in JOBS]}
save_status()

## 2) 사전 점검 — 틀리면 여기서 멈춘다

**이 셀만 일부러 예외를 던져 Run All 을 중단시킨다.** 8시간을 날리는 것보다 낫다.

- `bimamba_pure` 커맨드에 `--use_chunk_pairs` 가 **없어야** 한다
- `bimamba_pure` 커맨드에 `--policy.sscp_enabled=false` 가 **있어야** 한다
- 모든 잡의 커맨드에 `--seed={s}` 가 **있어야** 한다
- 출력 경로에 `seed{s}` 가 **있어야** 한다 (seed 0 체크포인트를 덮어쓰지 않는지)

In [ ]:
problems = []
for t, s, task in JOBS:
    cmd = v23.make_train_cmd(t, s, task, gpu_id=GPUS[0])
    ok_seed = f'--seed={s}' in cmd
    ok_dir  = f'seed{s}' in cmd
    print(f'--- {t}  seed{s} ---')
    print('  seed 플래그:', ok_seed, '| 출력 경로 seed 표기:', ok_dir)
    if not ok_seed:
        problems.append(f'{t} seed{s}: --seed={s} 가 커맨드에 없다')
    if not ok_dir:
        problems.append(f'{t} seed{s}: 출력 경로에 seed{s} 가 없다 (seed 0 을 덮어쓸 수 있다)')
    if t.startswith('bimamba_pure'):
        cp = '--use_chunk_pairs' in cmd
        sf = '--policy.sscp_enabled=false' in cmd
        print('  chunk_pairs 있음:', cp, '(False 여야) | sscp_enabled=false 있음:', sf, '(True 여야)')
        if cp:
            problems.append(f'{t} seed{s}: 순수 BiMamba 인데 --use_chunk_pairs 가 있다')
        if not sf:
            problems.append(f'{t} seed{s}: 순수 BiMamba 인데 --policy.sscp_enabled=false 가 없다')

STATUS['preflight'] = {'ok': not problems, 'problems': problems}
save_status()
if problems:
    raise AssertionError('사전 점검 실패 — 학습을 시작하지 않는다:\n  ' + '\n  '.join(problems))
print('\n사전 점검 통과.')

## 3) 단계 A — diag3 (`b_k` 변형, 학습 없음)

`mean` 의 회복률이 답이다. **낮으면 "위치마다 다르다" 가 이득의 원천**이다
(diag2 의 위치 분리 해석 확정). 변형은 전부 OOD 라 **비교만** 읽는다.

수 분이면 끝나고 **학습 시작 전에 돌리므로 GPU 를 안 뺏는다.**

In [ ]:
DIAG_SCRIPT = REPO / 'notebooks' / 'libero' / 'diag3_bk_variants.py'
DIAG_ENV = dict(os.environ,
                PYTHONPATH=str(REPO / 'src'),
                HF_HUB_DISABLE_XET='1',
                MPLBACKEND='Agg',
                CUDA_VISIBLE_DEVICES=DIAG_GPU)

def run_diag3():
    json_path = SHARE / f'diag3_all_{STAMP}.json'
    cmd = [v23.PYTHON, str(DIAG_SCRIPT), '--tags', 'all', '--seed', str(DIAG_SEED),
           '--step', str(DIAG_STEP), '--task', X.TASK, '--batch', '4',
           '--json', str(json_path)]
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=DIAG_ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    print(f'\n[exit {rc}]')
    res = json.loads(json_path.read_text(encoding='utf-8')) if json_path.exists() else {}
    STATUS['diag3'] = {'exit': rc, 'json': str(json_path)}
    # 서브프로세스가 죽어도 예외가 안 나므로 여기서 직접 실패로 올린다 (상태 파일이 OK 로 거짓말하지 않게).
    if rc not in (0, 2) or not res:
        raise RuntimeError(f'diag3 이 결과 없이 끝났다 (exit {rc}). 위 로그의 Traceback 을 볼 것.')
    bad = [t for t, r in res.items() if not r.get('ok')]
    if bad:
        print(f'!! 일부 태그 실패: {bad}')
    return res

diag3_res = stage('A: diag3', run_diag3) or {}

In [ ]:
# diag3 요약 — mean/shuffle/random_ortho 의 회복률 (zero=0%, real=100%)
def _diag3_table():
    ok = [(t, r) for t, r in diag3_res.items() if r.get('ok')]
    if not ok:
        print('diag3 결과 없음 (위 단계 로그 확인).')
        return
    print(f"{'tag':<20} {'K':>4} {'mean':>7} {'shuffle':>9} {'rand_ortho':>11} {'α=0.5':>7} {'α=2':>6}")
    print('-' * 70)
    summ = {}
    for tag, r in sorted(ok, key=lambda kv: kv[1]['K']):
        v = r['variants']
        span = v['zero']['err_mean'] - v['real']['err_mean']
        rec = lambda k: ((v['zero']['err_mean'] - v[k]['err_mean']) / span
                         if (k in v and span > 1e-9) else float('nan'))
        row = {k: rec(k) for k in ('mean', 'shuffle', 'random_ortho', 'alpha=0.5', 'alpha=2')}
        summ[tag] = {'K': r['K'], **row}
        print(f"{tag:<20} {r['K']:>4} {row['mean']:>7.0%} {row['shuffle']:>9.0%} "
              f"{row['random_ortho']:>11.0%} {row['alpha=0.5']:>7.0%} {row['alpha=2']:>6.0%}")
    STATUS['diag3_recovery'] = summ
    save_status()
    ms = [x['mean'] for x in summ.values()]
    print('\n읽는 법: mean 이 낮을수록 "위치마다 다르다" 가 이득의 원천이다.')
    if ms and max(ms) < 0.25:
        print('>> mean 회복률이 전 K 에서 25% 미만 — 위치 분리 해석이 굳는다.')
    elif ms and min(ms) > 0.75:
        print('>> mean 회복률이 높다 — 위치와 무관한 큰 벡터로 설명된다. 해석을 다시 볼 것.')
    print('⚠️ 모든 변형이 OOD 다. 절대 수치가 아니라 변형끼리의 비교만 읽을 것.')

stage('A2: diag3 요약', _diag3_table)

## 4) 단계 B — seed 학습 (4잡 × 4 GPU, ~8h)

**이 셀이 ~8시간 블로킹된다.** 이미 150k 인 잡은 skip, 중단된 것(PART)은 resume 한다.
중간에 죽었으면 **노트북을 다시 Run All 하면 이어서 간다.**

In [ ]:
def _train():
    return cf.run_training_jobs(JOBS, GPUS, prefetch_task=X.TASK)

stage('B: 학습', _train)

# 학습 결과 점검 — 어느 체크포인트가 실제로 생겼나
def _ckpt_report():
    rep = {}
    for t, s, task in JOBS:
        d = v23.train_dir(t, s, task) / 'checkpoints'
        steps = sorted(int(x.name) for x in d.iterdir() if x.name.isdigit()) if d.is_dir() else []
        rep[f'{t}/seed{s}'] = steps[-1] if steps else None
        print(f'  {t:<22} seed{s}  최신 step = {steps[-1] if steps else "없음"}')
    STATUS['ckpt_after_train'] = rep
    save_status()
    miss = [k for k, v in rep.items() if v != cf.CKPT_STEP]
    if miss:
        print(f'\n!! {cf.CKPT_STEP} 에 못 미친 잡: {miss} — eval 에서 자동 제외된다.')

stage('B2: 체크포인트 점검', _ckpt_report)

## 5) 단계 C — eval (seed 별)

`run_evals` 는 모듈 레벨 `SEED` 를 쓰므로 seed 마다 갈아 끼우며 순차로 돈다.
체크포인트가 없는 셀은 자동으로 빠지고, 끝난 셀은 skip 된다.
**끝나면 `SEED` 를 원래대로 되돌린다.**

In [ ]:
EVAL_JOBS = [X._rm_job(v, K, X.MAIN_STRIDE) for v in VARIANTS]
print('eval 셀:', [j['out'] for j in EVAL_JOBS])

def _eval_all():
    orig = X.SEED
    try:
        for s in SEEDS:
            print('\n' + '#' * 60 + f'\n# eval seed {s}\n' + '#' * 60)
            X.SEED = s
            try:
                X.run_evals(EVAL_JOBS, GPUS)
            except Exception as e:                    # 한 seed 가 죽어도 다음 seed 는 간다
                traceback.print_exc()
                print(f'!! seed {s} eval 실패: {type(e).__name__}: {e}')
                STATUS.setdefault('eval_errors', {})[str(s)] = f'{type(e).__name__}: {e}'
    finally:
        X.SEED = orig
        print('\nSEED 복구 ->', X.SEED)

stage('C: eval', _eval_all)

## 6) 단계 D — 집계 (gap 이 재현되는가)

**이 표가 오늘 밤의 답이다.** seed 0 은 기존 결과(`65.0 / 56.8`)를 그대로 읽는다.

In [ ]:
import statistics as st

def _aggregate():
    rows = []
    orig = X.SEED
    try:
        for s in [0] + SEEDS:
            X.SEED = s
            bi = X._sr(f'rm_bimamba_k{K}_s{X.MAIN_STRIDE}')
            ac = X._sr(f'rm_acm2_k{K}_s{X.MAIN_STRIDE}')
            rows.append({'seed': s, 'bimamba': bi, 'acm2': ac,
                         'gap': (bi - ac) if (bi is not None and ac is not None) else None})
    finally:
        X.SEED = orig

    f = lambda x: f'{x:.1f}' if x is not None else '  --'
    print(f"{'seed':>5} {'BiMamba':>9} {'ACM2':>8} {'gap':>8}")
    print('-' * 33)
    for r in rows:
        print(f"{r['seed']:>5} {f(r['bimamba']):>9} {f(r['acm2']):>8} {f(r['gap']):>8}")

    gaps = [r['gap'] for r in rows if r['gap'] is not None]
    summary = {'rows': rows}
    if len(gaps) >= 2:
        m, sd = st.mean(gaps), st.stdev(gaps)
        summary.update(gap_mean=m, gap_std=sd, n=len(gaps))
        print('-' * 33)
        print(f"{'평균':>5} {'':>9} {'':>8} {m:>8.1f}")
        print(f"{'표준편차':>5} {'':>9} {'':>8} {sd:>8.1f}")
        if sd < 0.3 * abs(m):
            verdict = f'재현된다 (gap {m:.1f} ± {sd:.1f}). 헤드라인 유지.'
        else:
            verdict = f'흔들린다 (gap {m:.1f} ± {sd:.1f}). seed 를 늘리거나 주장을 약하게 할 것.'
        summary['verdict'] = verdict
        print('\n>>', verdict)
    else:
        summary['verdict'] = 'seed 가 하나뿐이다 — 학습/eval 이 덜 끝났는지 확인할 것.'
        print('\n>>', summary['verdict'])

    STATUS['aggregate'] = summary
    save_status()
    (SHARE / f'summary_{STAMP}.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
    return summary

agg = stage('D: 집계', _aggregate)

## 7) 최종 상태

In [ ]:
print('단계별 결과')
print('-' * 50)
for name, s in STATUS['stages'].items():
    mark = 'OK ' if s['ok'] else 'FAIL'
    print(f"  [{mark}] {name:<24} {s['sec']:>7}s" + ('' if s['ok'] else f"   {s.get('error', '')}"))
print()
if agg:
    print('gap 판정 :', agg.get('verdict'))
print('상태 파일:', STATUS_PATH)
print('요약 파일:', SHARE / f'summary_{STAMP}.json')

## 8) 아침에 볼 것

### 먼저 — 브라우저를 닫았다면

셀 출력이 안 남아 있을 수 있다. **파일부터 본다:**

```
outputs/final/share/overnight/status_*.json    단계별 성공/실패, 체크포인트 step, diag3 회복률
outputs/final/share/overnight/summary_*.json   seed 별 성공률과 gap 판정
outputs/final/share/overnight/diag3_all_*.json diag3 원본
```

### 결과 읽는 법

| gap 결과 | 뜻 |
|---|---|
| 평균 ~8, 표준편차 < 2.5 | **재현. 헤드라인 유지.** 논문 제일 약한 곳이 사라진다 |
| 표준편차가 평균의 30% 이상 | 흔들린다. seed 를 늘리거나 주장을 약하게 |
| seed 1·2 에서 gap 이 반토막 | **제출 전에 알아서 다행인 경우.** 주장을 다시 잡아야 한다 |

어느 쪽이 나오든 **알고 내는 것과 모르고 내는 것은 다르다.**

### diag3 읽는 법

- **`mean` 회복률 < 25%** → 이득은 거의 전부 위치별 변화에서 온다 (diag2 해석 확정)
- **`mean` 회복률 > 75%** → 위치와 무관한 큰 벡터로 설명된다 (해석 재검토)
- `shuffle` ≈ `random_ortho` → 내용보다 "갈라진다" 가 본질
- 둘 다 `real` 보다 한참 높음 → 어느 위치에 어느 벡터가 가는지도 중요
- ⚠️ 변형은 전부 OOD — **절대 수치가 아니라 변형끼리의 비교만.**

### 중간에 죽었다면

이 노트북을 **다시 Run All** 하면 된다. 150k 인 학습은 skip, 중단된 건 resume,
끝난 eval 은 skip 이다. 단 셀 4(사전 점검)는 매번 다시 돈다 (몇 초).

### 다음 라운드 후보

1. `acm2_k150` · `acm2_k50` 을 150k 로 — 표의 55.6 / 45.4 는 150k 가 아니다 (diag2 에서 없다고 나옴)
2. E2 `bimamba_scan="random"` — `ecd-bimamba` 머지 + `VARIANTS` 등록 필요
3. K=150 seed 1·2

`JOBS` 와 `EVAL_JOBS` 만 바꾸면 이 노트북을 그대로 재사용한다.